In [2]:
pip install polars pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import polars as pl
import os

# Define file paths
DATA_DIR = "." # Adjust if your raw files are in a different folder
OUT_DIR = "./processed"

os.makedirs(OUT_DIR, exist_ok=True)

# 1. Define the exact columns we need from each massive file
submission_cols = [
    "accessionNumber", 
    "issuerCik", 
    "filingDate"
]

reporting_owner_cols = [
    "accessionNumber", 
    "rptOwnerCik", 
    "isDirector", 
    "isOfficer", 
    "isTenPercentOwner"
]

nonderiv_cols = [
    "accessionNumber", 
    "securityTitle", 
    "transactionDate", 
    "transactionCode", 
    "transactionShares", 
    "transactionPricePerShare", 
    "transactionAcquiredDisposedCode", 
    "directOrIndirectOwnership"
]

print("Scanning files and building lazy execution plan...")

# 2. Use lazy scanning (pl.scan_csv) so we don't blow up our RAM
# We enforce schema overrides for tricky columns if necessary, but standard CSV scanning 
# usually works well when we select specific columns immediately.
lf_submission = pl.scan_csv(
    os.path.join(DATA_DIR, "lit_submission.csv"), 
    ignore_errors=True
).select(submission_cols)

lf_owner = pl.scan_csv(
    os.path.join(DATA_DIR, "lit_reportingowner.csv"), 
    ignore_errors=True
).select(reporting_owner_cols)

lf_nonderiv = pl.scan_csv(
    os.path.join(DATA_DIR, "lit_nonderiv.csv"), 
    ignore_errors=True
).select(nonderiv_cols)

# 3. Perform the Relational Joins
# We use an inner join because a transaction without an issuer or an owner is useless for our graph.
print("Defining graph joins on accessionNumber...")
lf_joined = (
    lf_nonderiv
    .join(lf_submission, on="accessionNumber", how="inner")
    .join(lf_owner, on="accessionNumber", how="inner")
)

# 4. Clean and filter the temporal anchor
# We MUST have a valid transactionDate and core IDs to build the graph.
lf_cleaned = lf_joined.drop_nulls(
    subset=["transactionDate", "issuerCik", "rptOwnerCik"]
)

# Optional: Add a simple date filter here to strip out clearly erroneous historical dates 
# (e.g., typos in SEC filings claiming a trade happened in 1905).
# Assuming format is YYYY-MM-DD
lf_cleaned = lf_cleaned.filter(
    pl.col("transactionDate") >= "2010-01-01"
)

# 5. Execute the plan and sink to a highly compressed Parquet file
output_path = os.path.join(OUT_DIR, "ml_ready_transactions.parquet")

print("Executing plan and streaming to Parquet (this may take a few minutes)...")
lf_cleaned.sink_parquet(output_path)

print(f"Success! Highly compressed, ML-ready dataset saved to: {output_path}")

# 6. Verify the output
df_verify = pl.read_parquet(output_path)
print(f"\nFinal Dataset Shape: {df_verify.shape}")
print(df_verify.head(3))

Scanning files and building lazy execution plan...
Defining graph joins on accessionNumber...
Executing plan and streaming to Parquet (this may take a few minutes)...
